In [10]:
from pathlib import Path
from collections import Counter

import torch
from torch.utils.data import  Dataset,DataLoader, Subset
from torchvision import datasets, transforms
from PIL import Image
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt

In [11]:
project_dir = Path.cwd().parent

data_dir = project_dir / "data" / "train - merge - clean"

print("Project directory:", project_dir)
print("Dataset directory:", data_dir)
print("Dataset exists:", data_dir.exists())

Project directory: c:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم
Dataset directory: c:\Users\ASUS\Desktop\مکتب شریف\مکتب 145\پروژه دوم\data\train - merge - clean
Dataset exists: True


In [12]:
classes = sorted([p.name for p in data_dir.iterdir() if p.is_dir()])

print("Classes:", classes)
print("Number of classes:", len(classes))

Classes: ['ambulance', 'autobus', 'kamyun', 'kamyunet', 'minibus', 'savari', 'taxi', 'vanet']
Number of classes: 8


In [13]:
class_counts = {}

for class_name in classes:
    class_dir = data_dir / class_name

    images = [
        file for file in class_dir.iterdir()
        if file.is_file()
    ]

    class_counts[class_name] = len(images)

print(class_counts)

{'ambulance': 217, 'autobus': 249, 'kamyun': 201, 'kamyunet': 287, 'minibus': 224, 'savari': 248, 'taxi': 247, 'vanet': 249}


In [37]:
class VehicleDataset(Dataset):
    def __init__(self, image_paths,labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform


    def __len__(self):
        return len(self.image_paths)


    def __getitem__(self,index):
        image = Image.open(self.image_paths[index]).convert("RGB")
        label = self.labels[index]

        if self.transform is not None:
            image = self.transform(image)
        return image , label

In [40]:
import pickle
with open("vehicle_split.pkl","rb")as f:
    split_data = pickle.load(f)

In [41]:
X_train = split_data["X_train"]
X_val = split_data["X_val"]
y_train = split_data["y_train"]
y_val = split_data["y_val"]

In [42]:
# ساخت دیتاست با داده های تست و ولیدیشن

train_dataset = VehicleDataset(
    image_paths=X_train,
    labels=y_train
)


val_dataset = VehicleDataset(
    image_paths= X_val,
    labels= y_val
)

In [44]:
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

image, label = train_dataset[0]

print("Image type:", type(image))
print("Image size:", image.size)
print("Label:", label)

Train samples: 1537
Validation samples: 385
Image type: <class 'PIL.Image.Image'>
Image size: (303, 377)
Label: 3


In [45]:
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [47]:
train_dataset = VehicleDataset(
    image_paths=X_train,
    labels=y_train,
    transform=transform
)


val_dataset = VehicleDataset(
    image_paths=X_val,
    labels=y_val,
    transform=transform
)

In [48]:
image, label = train_dataset [0]
print("Image shape:", image.shape)
print("Image dtype:", image.dtype)
print("Label:", label)
print("Min pixel value:", image.min().item())
print("Max pixel value:", image.max().item())

Image shape: torch.Size([3, 224, 224])
Image dtype: torch.float32
Label: 3
Min pixel value: 0.0
Max pixel value: 1.0


In [49]:
#data loader

from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True,
    num_workers=0
)


val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0
)

In [50]:
images, labels = next(iter(train_loader))
print("Images shape:", images.shape)
print("Labels shape:", labels.shape)
print("Images dtype:", images.dtype)
print("Labels dtype:", labels.dtype)

Images shape: torch.Size([32, 3, 224, 224])
Labels shape: torch.Size([32])
Images dtype: torch.float32
Labels dtype: torch.int64
